# Genotox Pipeline — Launcher Notebook

이 노트북은 **얇은 launcher** 역할만 합니다.  
모든 로직은 `.py` 스크립트에 있으며, 이 노트북은 실행/모니터링용입니다.

## 파이프라인 구조 (7 Steps)
1. `step1_data_audit.py` — Raw 데이터 감사 & 스키마 통합
2. `step2_feature_build.py` — Broad feature table 생성
3. `step3_train.py` — Leakage-free CV 학습 & 평가
4. `step4_leakage_report.py` — Reviewer 방어용 leakage control report
5. `step5_ablation.py` — Ablation / Robustness study (논문용)
6. `step6_imbalance_study.py` — Imbalance 전략 비교
7. `step7_dashboard.py` — 종합 Dashboard 생성
8. `run_all.py` — 전체 파이프라인 마스터 실행기

## 0. 환경 확인

In [ ]:
import sys, os
print(f'Python: {sys.version}')

# 필수 패키지 확인
for pkg in ['pandas', 'numpy', 'sklearn', 'xgboost', 'rdkit', 'shap', 'imblearn', 'matplotlib', 'joblib']:
    try:
        __import__(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — pip install 필요')

## 1. 전체 파이프라인 실행 (권장)

In [ ]:
# Shortlist만 실행 (Ames:none+XGB, invitro:alert_bootstrap+XGB, invivo:alert_bootstrap+XGB)
!python run_all.py

In [ ]:
# 전체 조합 실행 (모든 endpoint × model × strategy)
# !python run_all.py --all

In [ ]:
# 논문용 FULL STUDY (ablation + imbalance + dashboard 포함)
# !python run_all.py --full-study

In [ ]:
# 전체 조합 + 논문용 full study
# !python run_all.py --all --full-study

## 2. 개별 단계 실행

In [ ]:
# Step 1: 데이터 감사만
!python step1_data_audit.py

In [ ]:
# Step 2: Feature table 빌드만
!python step2_feature_build.py

In [ ]:
# Step 3: 학습 (shortlist만)
!python step3_train.py

In [ ]:
# Step 3: 학습 (전체 조합)
# !python step3_train.py --all

In [ ]:
# Step 4: Leakage Control Report (Reviewer 방어)
!python step4_leakage_report.py

In [ ]:
# Step 5: Ablation / Robustness Study (논문용)
# !python step5_ablation.py

In [ ]:
# Step 6: Imbalance Strategy Comparison
# !python step6_imbalance_study.py

In [ ]:
# Step 7: Summary Dashboard
!python step7_dashboard.py

## 3. 결과 확인

In [ ]:
import pandas as pd
from pathlib import Path
import config as cfg

# Summary metrics
summary_path = cfg.SUMMARY_DIR / 'summary_metrics.csv'
if summary_path.exists():
    df = pd.read_csv(summary_path)
    cols = ['experiment_id', 'status', 'test_mcc', 'test_balanced_accuracy', 
            'test_roc_auc', 'test_sensitivity', 'test_specificity']
    available = [c for c in cols if c in df.columns]
    display(df[available])
else:
    print('summary_metrics.csv not found — run step3 first')

In [ ]:
# Best by endpoint
best_path = cfg.SUMMARY_DIR / 'best_by_endpoint.csv'
if best_path.exists():
    display(pd.read_csv(best_path))
else:
    print('best_by_endpoint.csv not found')

In [ ]:
# 아티팩트 파일 확인
for d in sorted(cfg.ARTIFACT_DIR.iterdir()):
    if d.is_dir():
        files = list(d.glob('*'))
        print(f'\n{d.name}/ ({len(files)} files)')
        for f in sorted(files)[:10]:
            print(f'  {f.name} ({f.stat().st_size/1024:.0f}KB)')

## 4. 시각화 갤러리

In [ ]:
from IPython.display import Image, display as ipy_display

for exp_dir in sorted(cfg.ARTIFACT_DIR.iterdir()):
    if not exp_dir.is_dir():
        continue
    pngs = sorted(exp_dir.glob('*.png'))
    if pngs:
        print(f'\n=== {exp_dir.name} ===')
        for png in pngs:
            print(f'\n{png.name}')
            ipy_display(Image(filename=str(png), width=500))